# Predicting Irrigation Need — Exploratory Data Analysis

**Competition:** [Kaggle Playground Series S6E4](https://kaggle.com/competitions/playground-series-s6e4)  
**Goal:** Predict irrigation need class (Low, Medium, High) from environmental conditions  
**Evaluation Metric:** Balanced Accuracy Score

---

## Notebook Structure
1. Imports & Data Loading
2. Dataset Overview
3. Target Variable Analysis
4. Missing Values
5. Numerical Feature Analysis
6. Categorical Feature Analysis
7. Correlation Analysis (Numerical)
8. Mixed Correlation Analysis (Numerical + Categorical)
9. Key Findings

## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from dython.nominal import associations

import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Adjust paths for local vs Kaggle environment
TRAIN_PATH = '../data/raw/train.csv'  # local
TEST_PATH  = '../data/raw/test.csv'

# TRAIN_PATH = '/kaggle/input/playground-series-s6e4/train.csv'  # Kaggle
# TEST_PATH  = '/kaggle/input/playground-series-s6e4/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')

## 2. Dataset Overview

In [ ]:
train_df.sample(5, random_state=42)

In [ ]:
train_df.info()

In [ ]:
train_df.describe().T

In [ ]:
num_cols = train_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

# Remove id and target from feature lists
num_cols = [c for c in num_cols if c not in ['id']]
cat_cols = [c for c in cat_cols if c not in ['Irrigation_Need']]

print(f'Numerical features ({len(num_cols)}): {num_cols}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

## 3. Target Variable Analysis

In [ ]:
print('Unique classes:', train_df['Irrigation_Need'].unique())
print()

class_counts = train_df['Irrigation_Need'].value_counts()
class_pct    = train_df['Irrigation_Need'].value_counts(normalize=True) * 100

class_summary = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pct.round(2)
})
print(class_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=train_df, x='Irrigation_Need', order=class_counts.index, ax=axes[0])
axes[0].set_title('Class Distribution — Count')
axes[0].set_xlabel('Irrigation Need')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(
    class_counts,
    labels=class_counts.index,
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Class Distribution — Proportion')

plt.tight_layout()
plt.show()

## 4. Missing Values

In [ ]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct.round(2)
}).query('`Missing Count` > 0')

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)

In [ ]:
msno.matrix(train_df)
plt.title('Missing Value Matrix')
plt.show()

## 5. Numerical Feature Analysis

In [ ]:
# Distribution of each numerical feature
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=train_df, x=col, hue='Irrigation_Need', kde=True, ax=axes[i], bins=40)
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel('')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions by Irrigation Need', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots — how each numerical feature separates the classes
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=train_df, x='Irrigation_Need', y=col, ax=axes[i])
    axes[i].set_title(f'{col} by Class')
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features vs Irrigation Need (Boxplots)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

In [ ]:
# Unique value counts per categorical feature
for col in cat_cols:
    print(f'{col}: {train_df[col].nunique()} unique values → {train_df[col].unique().tolist()}')

In [ ]:
# Class distribution within each categorical feature
n_cols_plot = 2
n_rows_plot = int(np.ceil(len(cat_cols) / n_cols_plot))

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(14, n_rows_plot * 5))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(train_df[col], train_df['Irrigation_Need'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], colormap='Set2')
    axes[i].set_title(f'{col} vs Irrigation Need (%)')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Percentage')
    axes[i].legend(loc='upper right', fontsize=8)
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Class Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Correlation Analysis — Numerical Features

In [ ]:
correlation_mat = train_df[num_cols].corr()

mask = np.zeros_like(correlation_mat)
mask[np.triu_indices_from(mask)] = True

plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_mat,
    vmax=1, vmin=-1,
    annot=True,
    annot_kws={'fontsize': 8},
    mask=mask,
    cmap=sns.diverging_palette(20, 220, as_cmap=True),
    linewidths=0.5
)
plt.title('Pearson Correlation — Numerical Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Highlight highly correlated pairs (|r| > 0.7)
high_corr = (
    correlation_mat
    .abs()
    .unstack()
    .sort_values(ascending=False)
    .drop_duplicates()
)
high_corr = high_corr[(high_corr > 0.7) & (high_corr < 1.0)]

if high_corr.empty:
    print('No highly correlated numerical feature pairs found (threshold: 0.7)')
else:
    print('Highly correlated pairs (|r| > 0.7):')
    print(high_corr)

## 8. Mixed Correlation Analysis — Numerical + Categorical

Using `dython` associations:
- **Pearson** for numerical-numerical pairs
- **Cramér's V** for categorical-categorical pairs  
- **Correlation ratio** for numerical-categorical pairs

In [ ]:
# Drop id before associations — it's not a feature
assoc_df = train_df.drop(columns=['id'])

associations(
    assoc_df,
    figsize=(16, 12),
    mark_columns=True,
    clustering=True,
    title='Mixed Feature Associations (Pearson / Cramér\'s V / Correlation Ratio)'
)
plt.tight_layout()
plt.show()

In [ ]:
# Association strength of each feature with the target
from dython.nominal import associations

assoc_result = associations(assoc_df, compute_only=True)
target_assoc = (
    assoc_result['corr']['Irrigation_Need']
    .drop('Irrigation_Need')
    .abs()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
target_assoc.plot(kind='barh', color='steelblue')
plt.title('Feature Association Strength with Irrigation Need')
plt.xlabel('Association Strength')
plt.axvline(x=0.1, color='red', linestyle='--', label='Threshold (0.1)')
plt.legend()
plt.tight_layout()
plt.show()

print(target_assoc)

## 9. Key Findings

> Fill this in after running the notebook. Use this section to summarize what you found before moving to modeling.

**Class Balance:**
- [ ] Classes are balanced / imbalanced (update after running)

**Missing Values:**
- [ ] No missing values / X% missing in Y columns (update after running)

**Strong Predictors (high association with target):**
- [ ] Feature A
- [ ] Feature B

**Highly Correlated Feature Pairs (potential redundancy):**
- [ ] Feature X and Feature Y (r = ?)

**Categorical Feature Observations:**
- [ ] Note any categorical features that strongly separate classes

**Modeling Implications:**
- [ ] Note any feature engineering ideas from EDA
- [ ] Note any features to consider dropping